# Flow trajectories in connectome space (spec v4 E2.1, second addendum)

What a trained flow does when it generates one subject's FC from their SC, shown in **edge space**.

The model is the seed-0 **un-anchored nonlinear flow** with **block FC PCA targets** (no CCA, no ridge anchor): it
starts from Gaussian noise $z_0$ in the 256 block PCA coordinates and integrates a learned velocity field
$v_\theta(z_\tau, \tau, \mathrm{SC}, m)$ to $\tau = 1$, conditioned on the subject's SC codes and the target condition
$m$. Every intermediate state is decoded to a full connectome,

$$
\mathrm{FC}_\tau = \mu_m + s_m\, W_m z_\tau ,
$$

where $\mu_m$ is condition $m$'s **group-mean connectome** (the backbone every prediction sits on) and $s_m W_m z_\tau$
the subject-level deviation in edge space. At $\tau = 0$ the deviation is pure noise on top of the backbone; at
$\tau = 1$ it should be the subject's predicted signature. Matrices are ordered by Yeo-7 network.

Sections: (1) setup; (2) one subject's trajectory, full FC and deviation from the group mean, next to their measured
FC and the ridge prediction; (3) the same for a few more subjects (deviations only); (4) how accuracy and
identifiability build up along the flow, over all 184 test subjects; (5) a subject's generated distribution: several
samples at $\tau = 1$.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "main.py").exists())
EXP = REPO_ROOT / "scripts" / "experiments" / "joint_flow_matching"
for p in (REPO_ROOT, EXP):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import e21_run as R
from models.eval.metrics import compute_corr_matrix, corr_avg_rank

SEED, MODEL, TARGET = 0, "flow_free", "pca"
COND = "rest"                       # target condition shown in sections 2-3
TAUS = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]
N_STEPS = 32                        # RK4 steps, as in training evaluation
torch.manual_seed(0)

device = torch.device("cpu")
data = R.Data(SEED, device, target=TARGET)
ckpt = torch.load(R.CACHE / "models" / f"{MODEL}_seed{SEED}_{TARGET}.pt", map_location="cpu")
hp = ckpt["hp"]
net = R.make(MODEL, hp, data.d)
net.load_state_dict(ckpt["state"])
net.eval()
m = R.FC.index(COND)
sd = data.L.sd
mu = sd.train_mean[COND]                                     # group-mean connectome (train)
test_fc = sd.edges[("test", COND)]
n_test = test_fc.shape[0]
print(f"seed {SEED}, {MODEL} on {TARGET} targets; hp {hp}; {n_test} test subjects; condition {COND}")

roi = pd.read_csv(REPO_ROOT / "data" / "atlas_info" / "Glasser_dseg_reformatted.csv")
nets_order = ["visual", "somatosensory", "dorsal attention", "ventral attention", "limbic", "frontoparietal", "default mode"]
net_lab = roi["community_yeo"].fillna("other").to_numpy()
order = np.concatenate([np.where(net_lab == n)[0] for n in nets_order + ["other"]])
bounds = np.cumsum([np.sum(net_lab == n) for n in nets_order + ["other"]])[:-1]
P = len(roi)
iu = np.triu_indices(P, k=1)

def to_mat(edges):
    M = np.zeros((P, P))
    M[iu] = edges
    M = M + M.T
    return M[np.ix_(order, order)]

def show(ax, edges, vmax, title, cmap="RdBu_r"):
    ax.imshow(to_mat(edges), cmap=cmap, vmin=-vmax, vmax=vmax, interpolation="nearest")
    for b in bounds:
        ax.axhline(b - 0.5, color="k", lw=0.3); ax.axvline(b - 0.5, color="k", lw=0.3)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=9)

def dr(a, b):
    a, b = a - mu, b - mu
    a, b = a - a.mean(), b - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

In [ ]:
@torch.no_grad()
def trajectory(subjects, n_samples=1, seed=0):
    # integrate from noise for the given test subjects; returns {tau: edges (n_samples, len(subjects), E)}
    g = torch.Generator().manual_seed(seed)
    subj = torch.as_tensor(subjects).repeat(n_samples)
    cond = data.sc["test"][subj, :hp["k_sc"]]
    tok = torch.full((len(subj),), m)
    z = torch.randn((len(subj), data.d), generator=g)
    keep, h = {}, 1.0 / N_STEPS
    v = lambda x, t: net(x, tok, tok, torch.full((len(x),), t), cond=cond)
    for i in range(N_STEPS + 1):
        t = i * h
        for tk in TAUS:
            if abs(t - tk) < h / 2:
                keep[tk] = z.clone()
        if i == N_STEPS:
            break
        k1 = v(z, t); k2 = v(z + 0.5 * h * k1, t + 0.5 * h); k3 = v(z + 0.5 * h * k2, t + 0.5 * h); k4 = v(z + h * k3, t + h)
        z = z + (h / 6) * (k1 + 2 * k2 + 2 * k3 + k4)
    out = {}
    for tk, zt in keep.items():
        raw = (zt * data.s_lat).numpy()
        out[tk] = data.L.decode(m, raw).reshape(n_samples, len(subjects), -1)
    return out

ridge_pred = data.L.decode(m, data.rc_raw["test"][m])          # ridge (per condition) prediction, edges

## 2. One subject: from noise on the group-mean backbone to their signature

Top row: the full connectome at each flow time (a single sample), the subject's measured FC, and the group mean.
Bottom row: the same as deviations from the group mean ($\mathrm{FC} - \mu$), which is where the subject-specific
signal lives; demeaned r with the subject's measured FC in each title.

In [ ]:
subj = 0
tr = trajectory([subj])
fig, axes = plt.subplots(2, len(TAUS) + 2, figsize=(2.6 * (len(TAUS) + 2), 5.6), layout="constrained")
for j, tk in enumerate(TAUS):
    e = tr[tk][0, 0]
    show(axes[0, j], e, 0.8, f"τ = {tk:g}")
    show(axes[1, j], e - mu, 0.25, f"τ = {tk:g}\ndemeaned r {dr(e, test_fc[subj]):.2f}")
show(axes[0, -2], test_fc[subj], 0.8, "measured FC")
show(axes[1, -2], test_fc[subj] - mu, 0.25, "measured − group mean")
show(axes[0, -1], mu, 0.8, "group mean (backbone)")
show(axes[1, -1], ridge_pred[subj] - mu, 0.25, f"ridge prediction\ndemeaned r {dr(ridge_pred[subj], test_fc[subj]):.2f}")
axes[0, 0].set_ylabel("full FC"); axes[1, 0].set_ylabel("FC − group mean")
fig.suptitle(f"Test subject {subj}: SC → {COND} FC, flow from noise (seed {SEED}, block PCA targets)", fontweight="bold")

## 3. More subjects (deviations from the group mean)

Each row is one test subject: deviation at increasing flow time, then the mean of 32 samples at $\tau = 1$, then the
measured deviation. The scale of the predicted deviation is much smaller than the measured one: SC predicts only a
small part of each subject's FC (E2.1), so the generated signatures are shrunk toward the group.

In [ ]:
subjects = [1, 2, 3, 4]
tr = trajectory(subjects)
tr_mean = trajectory(subjects, n_samples=32, seed=1)[1.0].mean(0)
fig, axes = plt.subplots(len(subjects), len(TAUS) + 2, figsize=(2.4 * (len(TAUS) + 2), 2.4 * len(subjects)), layout="constrained")
for i, s in enumerate(subjects):
    for j, tk in enumerate(TAUS):
        show(axes[i, j], tr[tk][0, i] - mu, 0.25, f"τ = {tk:g}" if i == 0 else "")
    show(axes[i, -2], tr_mean[i] - mu, 0.08, f"mean of 32, r {dr(tr_mean[i], test_fc[s]):.2f}")
    show(axes[i, -1], test_fc[s] - mu, 0.25, "measured" if i == 0 else "")
    axes[i, 0].set_ylabel(f"subject {s}")
fig.suptitle("Deviation from the group mean along the flow (single sample), the sample mean (note the narrower scale) "
             "and the measured deviation", fontweight="bold", fontsize=11)

## 4. How subject information builds up along the flow (all 184 test subjects)

For each flow time: demeaned r between a generated sample and the subject's measured FC; the same for the mean of 16
samples; and identification (average rank of the subject's own measured FC among all 184, for the sample mean).

In [ ]:
all_subj = list(range(n_test))
trs = trajectory(all_subj, n_samples=16, seed=2)
rows = []
for tk in TAUS:
    single = np.mean([dr(trs[tk][0, i], test_fc[i]) for i in all_subj])
    mean16 = trs[tk].mean(0)
    mdr = np.mean([dr(mean16[i], test_fc[i]) for i in all_subj])
    cc = compute_corr_matrix(test_fc - mu, mean16 - mu)
    rows.append(dict(tau=tk, single_sample_dr=single, mean16_dr=mdr, mean16_avg_rank=float(corr_avg_rank(cc=cc)),
                     deviation_norm_ratio=float(np.linalg.norm(trs[tk][0] - mu, axis=1).mean()
                                                / np.linalg.norm(test_fc - mu, axis=1).mean())))
traj_tab = pd.DataFrame(rows)
display(traj_tab.round(4))
print(f"ridge reference: demeaned r {np.mean([dr(ridge_pred[i], test_fc[i]) for i in all_subj]):.4f}")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), layout="constrained")
axes[0].plot(traj_tab.tau, traj_tab.single_sample_dr, "o-", label="single sample")
axes[0].plot(traj_tab.tau, traj_tab.mean16_dr, "o-", label="mean of 16 samples")
axes[0].set_ylabel("demeaned r with measured FC"); axes[0].legend(frameon=False)
axes[1].plot(traj_tab.tau, traj_tab.mean16_avg_rank, "o-", color="C2"); axes[1].axhline(0.5, color="grey", ls="--")
axes[1].set_ylabel("identification (avg rank; chance 0.5)")
axes[2].plot(traj_tab.tau, traj_tab.deviation_norm_ratio, "o-", color="C3")
axes[2].set_ylabel("deviation size: generated / measured")
for ax in axes: ax.set_xlabel("flow time τ")
fig.suptitle(f"Along the flow, test subjects (seed {SEED}, {COND})", fontweight="bold")

## 5. One subject's generated distribution at $\tau = 1$

Four independent samples for the same subject (same SC, different starting noise), their mean, and the measured
deviation: the spread between samples is what the flow believes SC leaves undetermined.

In [ ]:
subj = 0
samp = trajectory([subj], n_samples=4, seed=3)[1.0][:, 0]
fig, axes = plt.subplots(1, 6, figsize=(16, 3.0), layout="constrained")
for j in range(4):
    show(axes[j], samp[j] - mu, 0.25, f"sample {j + 1}, r {dr(samp[j], test_fc[subj]):.2f}")
show(axes[4], samp.mean(0) - mu, 0.12, f"mean of 4, r {dr(samp.mean(0), test_fc[subj]):.2f}")
show(axes[5], test_fc[subj] - mu, 0.25, "measured")
fig.suptitle(f"Test subject {subj}: four generated {COND} connectomes (deviation from the group mean)", fontweight="bold")